In [2]:
# train.csv 컬럼 설명
# msno      : 유저 고유 ID (2017년 2월 만료 코호트, PK)
# is_churn  : 타겟 변수. 만료 후 30일 이내 재구독 안 하면 1(이탈), 재구독하면 0(잔존)

# train_v2.csv 컬럼 설명
# train.csv와 컬럼 구조는 동일하지만, 2017년 3월 만료 코호트를 담은 별도 스냅샷
# msno      : 유저 고유 ID (2017년 3월 만료 코호트, PK)
# is_churn  : 타겟 변수 (정의는 train.csv와 동일)

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# 시각화 스타일 (팔레트: series-1 blue #2a78d6, series-2 orange #eb6834, 그리드 #e1e0d9)
# sns.set_style를 먼저 호출한 뒤 rcParams.update를 마지막에 적용해야
# seaborn이 font.family를 sans-serif로 되돌리는 것을 막을 수 있음
sns.set_style("whitegrid", {"grid.color": "#e1e0d9", "axes.edgecolor": "#c3c2b7"})
plt.rcParams.update({
    "figure.dpi": 110,
    "font.family": "Malgun Gothic",
    "axes.unicode_minus": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b",
    "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#898781",
    "ytick.color": "#898781",
    "grid.color": "#e1e0d9",
    "text.color": "#0b0b0b",
})

SERIES_1 = "#2a78d6"  # train (2월 코호트)
SERIES_2 = "#eb6834"  # train_v2 (3월 코호트)
BLUE_SEQ_CMAP = LinearSegmentedColormap.from_list(
    "series1_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281"]
)
pd.set_option("display.max_columns", None)

In [ ]:
train = pd.read_csv("../data/raw/train.csv")
train_v2 = pd.read_csv("../data/raw/train_v2.csv")

print("train:", train.shape)
print(train.dtypes)
display(train.head())

print("\ntrain_v2:", train_v2.shape)
print(train_v2.dtypes)
display(train_v2.head())

train: (992931, 2)
msno        object
is_churn     int64
dtype: object


,msno,is_churn
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1


NameError: name 'train_v2' is not defined

In [6]:
print(f"train msno 중복 존재 여부: {train['msno'].duplicated().any()}")
print(f"train_v2 msno 중복 존재 여부: {train_v2['msno'].duplicated().any()}")
print()
print("train 결측치:"); print(train.isna().sum())
print("\ntrain_v2 결측치:"); print(train_v2.isna().sum())

train msno 중복 존재 여부: False


NameError: name 'train_v2' is not defined

In [7]:
print("train is_churn 비율:")
print(train["is_churn"].value_counts(normalize=True))
print("\ntrain_v2 is_churn 비율:")
print(train_v2["is_churn"].value_counts(normalize=True))

train is_churn 비율:
is_churn
0    0.936077
1    0.063923
Name: proportion, dtype: float64

train_v2 is_churn 비율:


NameError: name 'train_v2' is not defined

In [8]:
categories = ["0 (잔존)", "1 (이탈)"]
x = np.arange(len(categories))
width = 0.35

train_rates = train["is_churn"].value_counts(normalize=True).sort_index().values * 100
train_v2_rates = train_v2["is_churn"].value_counts(normalize=True).sort_index().values * 100

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width / 2, train_rates, width, label="train (2월 코호트)", color=SERIES_1)
ax.bar(x + width / 2, train_v2_rates, width, label="train_v2 (3월 코호트)", color=SERIES_2)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("비율 (%)")
ax.set_title("is_churn 분포 비교 (train vs train_v2)")
ax.legend()
ax.grid(axis="y")
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

NameError: name 'train_v2' is not defined

In [9]:
train_msno = set(train["msno"])
train_v2_msno = set(train_v2["msno"])
overlap_msno = train_msno & train_v2_msno

train_only_n = len(train_msno - train_v2_msno)
overlap_n = len(overlap_msno)
train_v2_only_n = len(train_v2_msno - train_msno)

print(f"train만: {train_only_n:,} ({train_only_n / len(train_msno) * 100:.1f}% of train)")
print(f"겹침: {overlap_n:,} ({overlap_n / len(train_msno) * 100:.1f}% of train, {overlap_n / len(train_v2_msno) * 100:.1f}% of train_v2)")
print(f"train_v2만: {train_v2_only_n:,} ({train_v2_only_n / len(train_v2_msno) * 100:.1f}% of train_v2)")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(["train만", "겹침(두 코호트 모두)", "train_v2만"], [train_only_n, overlap_n, train_v2_only_n], color=SERIES_1, width=0.5)
ax.set_ylabel("user count")
ax.set_title("train / train_v2 유저 중복 현황")
ax.grid(axis="y")
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

NameError: name 'train_v2' is not defined

In [10]:
merged = train.merge(train_v2, on="msno", suffixes=("_feb", "_mar"))
ct = pd.crosstab(merged["is_churn_feb"], merged["is_churn_mar"])

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(ct.values, cmap=BLUE_SEQ_CMAP)
ax.set_xticks([0, 1]); ax.set_xticklabels(categories)
ax.set_yticks([0, 1]); ax.set_yticklabels(categories)
ax.set_xlabel("is_churn (3월 코호트, train_v2)")
ax.set_ylabel("is_churn (2월 코호트, train)")
ax.set_title(f"중복 유저({len(merged):,}명)의 라벨 일치 현황")
vmax = ct.values.max()
for i in range(2):
    for j in range(2):
        val = ct.values[i, j]
        text_color = "#ffffff" if val > vmax / 2 else "#0b0b0b"
        ax.text(j, i, f"{val:,}", ha="center", va="center", color=text_color)
plt.colorbar(im, ax=ax, label="user count")
plt.tight_layout()
plt.show()

NameError: name 'train_v2' is not defined

In [11]:
agree_rate = (merged["is_churn_feb"] == merged["is_churn_mar"]).mean()
flip_to_churn = ((merged["is_churn_feb"] == 0) & (merged["is_churn_mar"] == 1)).sum()
flip_to_retain = ((merged["is_churn_feb"] == 1) & (merged["is_churn_mar"] == 0)).sum()

print(f"라벨 일치율 (2월과 3월 is_churn이 같음): {agree_rate * 100:.1f}%")
print(f"2월 잔존 -> 3월 이탈로 전환: {flip_to_churn:,}명")
print(f"2월 이탈 -> 3월 잔존(winback)으로 전환: {flip_to_retain:,}명")

NameError: name 'merged' is not defined

In [ ]:
# ------------------------------------------------------------------
# 분할 전략에 주는 시사점 (전처리/모델링 단계에서 반영 필요)
# 1. train.csv와 train_v2.csv의 유저 중복이 88~91%로 매우 높음
#    -> 두 코호트를 합친 뒤 "2월=train / 3월=valid,test" 같은 시간 기반 분할을 쓰면,
#       같은 유저의 (거의 동일한 컷오프 기반) 피처가 train과 valid/test에 동시에 등장해
#       사실상 데이터 누수에 가까운 효과가 생김
#    -> 따라서 시간 기반이 아니라 msno(유저) 단위로 그룹 분할해야 함
#       (같은 유저의 모든 행은 반드시 하나의 split에만 속하도록)
# 2. is_churn 비율이 6~9%로 불균형 -> 분할 시 stratify 필요, 모델링 시 클래스 불균형 처리(가중치/샘플링 등) 고려
# 3. 겹치는 유저 중 5.3%는 두 시점 사이 라벨이 바뀜(이탈<->잔존 전환) -> 유저의 이탈 성향이 완전히 고정된 값이 아니라
#    시간에 따라 변할 수 있음을 시사 (모델 해석 시 참고)
# ------------------------------------------------------------------
print("분석 요약 완료 (위 주석 참고)")

분석 요약 완료 (위 주석 참고)
